In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/grandiosfotisemo/dataset-cours-terminale/dataset_cours_nettoye.csv
/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [7]:
df= pd.read_csv('/kaggle/input/datasets/grandiosfotisemo/dataset-cours-terminale/dataset_cours_nettoye.csv')
df.head()

,cours,titre_chapitre
0,Tale Specialite 1 D EFINITIONS Mathematiques :...,[C] continuite
1,Tale : Mathematiques expertes Arithmetique Mat...,[C]Arithmetique
2,Tale Specialite Mouvement des corps celestes P...,[C]celestes
3,Tale Specialite Cinematique du point materiel ...,[C]cinematique
4,Tale Specialite Convexite Mathematiques : Conv...,[C]Convexite


In [8]:
def chunk_text(text, chunk_size=1500, overlap=200):
    chunks = []

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [9]:
chunks = []
metadata = []

for idx, row in df.iterrows():

    cours_chunks = chunk_text(
        row["cours"],
        chunk_size=1500,
        overlap=200
    )

    for chunk_id, chunk in enumerate(cours_chunks):

        chunks.append(chunk)

        metadata.append({
            "document_id": idx,
            "titre_chapitre": row["titre_chapitre"],
            "chunk_id": chunk_id
        })

print("Nombre de documents :", len(df))
print("Nombre de chunks :", len(chunks))
print("Mean chunk / document :", len(chunks)/len(df))

Nombre de documents : 29
Nombre de chunks : 279
Mean chunk / document : 9.620689655172415


In [10]:
import torch
import transformers
import sentence_transformers

In [11]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda",
    model_kwargs={"torch_dtype": torch.float16}
)

print("compilation OK")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

compilation OK


In [12]:
embeddings = model.encode(
    chunks,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

In [37]:
import numpy as np

def rechercher_cours(question, top_k=3):

    # transforme la question en vecteur
    question_vecteur = model.encode(
        [question],
        prompt_name="query",
        normalize_embeddings=True
    )

    # comparaison avec les chunks
    scores = question_vecteur @ embeddings.T

    # indices des meilleurs résultats
    indices = np.argsort(scores[0])[::-1][:top_k]

    print("\nRésultats pour :", question)

    for i, idx in enumerate(indices):

        print("\n------------------------------")
        print("Résultat", i + 1)
        print("Score :", round(float(scores[0][idx]), 4))
        print("Chapitre :", metadata[idx]["titre_chapitre"])
        print("Chunk :", metadata[idx]["chunk_id"])
        print("\n", chunks[idx])

rechercher_cours("explique les matrices")


Résultats pour : explique les matrices

------------------------------
Résultat 1
Score : 0.7764
Chapitre : [C]Matrice
Chunk : 0

 Tale Maths Expertes 2 OP ERATIONS SUR LES MATRICES Mathematiques : Matrices et application aux graphes 1 Introduction aux Matrices Les matrices sont des objets mathematiques fondamentaux qui permettent de representer et de resoudre de nom- breux problemes issus de diverses disciplines. Elles constituent un outil puissant pour modeliser des situations complexes et effectuer des calculs de maniere efficace. Historiquement, l'utilisation de tableaux de nombres pour resoudre des systemes lineaires remonte a l'Antiquite, mais c'est Arthur Cayley qui, au milieu du XIXe siecle, a formalise la notion de matrice comme objet de calcul representant des transformations lineaires. Definition 1.Unematriceest un tableau rectangulaire de nombres reels disposes en lignes et en colonnes. Une matrice amlignes etncolonnes est appelee matrice de format (m, n) ou matricemn. On 

In [38]:
question = input("question : ")

rechercher_cours(
    question,
    top_k=3
)

question :  donne moi la définition de la fonction cosinus



Résultats pour : donne moi la définition de la fonction cosinus

------------------------------
Résultat 1
Score : 0.7114
Chapitre : [C]Trigo
Chunk : 3

 les fonctions trigonometriques mais selon une approche analytique : etude de fonction classique. On traite dans l'ordre les fonctions cos, sin et tan. 2 Approche analytique 2.1 La fonction cosinus La fonction cosinus est une fonction que l'on construit et qu'on a d'abord utilise en geometrie. Sa construction implique une serie de proprietes comme sa parite, sa periodicite, son caractere continue et derivable ou encore le fait qu'elle soit bornee (minoree et majoree). Remarque :(HP+)Malgre que cette fonction est une construction provenant d'abord de la geometrie, il est possible de l'exprimer comme un polynomeinfini de la maniere suivante, xR,cos(x) = +X n=0 (1)n (2n)! x2n Cette expression, bien que hors-programme, permet de voir comment les machines (calculatrice par exemple) calcule le cosinus d'un reel. Remarque :La factorielle d'u

In [22]:
!pip install -q groq

In [23]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

### choix modèle
LLM_MODEL = "openai/gpt-oss-120b"

<img src="https://console.groq.com/_next/static/media/openailogo.523c87a0.svg" width="20%">

In [24]:
import numpy as np
import json

def rechercher_cours_tool(query, top_k=3):

    # transforme la recherche en vecteur
    vecteur = model.encode(
        [query],
        prompt_name="query",
        normalize_embeddings=True
    )

    # compare avec tous les chunks
    scores = (vecteur @ embeddings.T)[0]
    indices = np.argsort(scores)[::-1][:top_k]

    resultats = []

    for idx in indices:
        resultats.append({
            "chapitre": metadata[idx]["titre_chapitre"],
            "chunk": metadata[idx]["chunk_id"],
            "texte": chunks[idx]
        })

    return json.dumps(resultats)

In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "rechercher_cours",
            "description": "Recherche des passages pertinents dans les cours.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Information à rechercher dans les cours."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [26]:
def demander(question):

    messages = [
        {"role": "system", "content": """
        Tu es un assistant pédagogique.
        Si la question concerne les cours, utilise l'outil rechercher_cours.
        Utilise les passages trouvés pour répondre correctement.
        """},
        {"role": "user", "content": question}
    ]

    # Partie RAG / outils
    for _ in range(3):

        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        message = response.choices[0].message

        # Le LLM ne demande plus d'outil
        if not message.tool_calls:
            texte_reponse = message.content
            break

        messages.append(message)

        # Exécution des outils demandés
        for call in message.tool_calls:

            if call.function.name == "rechercher_cours":

                args = json.loads(call.function.arguments)

                resultat = rechercher_cours_tool(
                    args["query"]
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "name": "rechercher_cours",
                    "content": resultat
                })

    else:
        return {"erreur": "Trop de recherches"}

    # Deuxième appel : uniquement pour produire le JSON final
    final = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                Transforme la réponse en JSON valide.

                Format attendu :
                {
                    "reponse": "réponse claire à l'utilisateur"
                }

                Retourne uniquement le JSON.
                """
            },
            {
                "role": "user",
                "content": texte_reponse
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(final.choices[0].message.content)

In [41]:
reponse = demander(
    "Donne moi l'énoncé du petit théorème de Fermat"
)

print(reponse)

{'reponse': 'En effet, le petit théorème de Fermat s’énonce ainsi : pour tout nombre premier p et tout entier n, on a n^{p}\\equiv n \\pmod{p}. Si p ne divise pas n, on peut simplifier pour obtenir n^{p-1}\\equiv 1 \\pmod{p}. La preuve classique repose sur le développement de (n+1)^{p} avec le binôme, les coefficients \\binom{p}{k} étant multiples de p pour 1\\le k\\le p-1, ce qui donne (n+1)^{p}\\equiv n^{p}+1 \\pmod{p}. En appliquant cette congruence par récurrence on obtient n^{p}\\equiv n \\pmod{p}. La seconde partie suit immédiatement en multipliant par l’inverse de n modulo p. Une preuve alternative utilise le fait que le groupe des unités (\\mathbb{Z}/p\\mathbb{Z})^{\\times} est cyclique d’ordre p-1, donc tout élément non nul satisfait a^{p-1}=1.'}
